# Modelling: predict the number of rings

First simple model, with experiments tracked in MLflow. MLflow data stays local (`mlflow.db` and `mlruns/` are git-ignored).

## Imports

In [7]:
import mlflow
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split

## Data

In [8]:
df = pd.read_csv("../data/abalone.csv")
df.head()

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


## Preprocessing

One-hot encode `Sex` and split features / target.

In [9]:
def preprocess(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Encode the Sex column and split the data into features and target."""
    df = pd.get_dummies(df, columns=["Sex"], dtype=int)
    X = df.drop(columns=["Rings"])
    y = df["Rings"]
    return X, y


X, y = preprocess(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((3341, 10), (836, 10))

## MLflow setup

In [10]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("abalone-age-prediction")

<Experiment: artifact_location='/Users/njibs/xhec-mlops-2026-project/notebooks/mlruns/1', creation_time=1792671272023, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792671272023, lifecycle_stage='active', name='abalone-age-prediction', tags={}, trace_location=None, workspace='default'>

## Train and log models

We compare a linear regression (baseline) and a random forest.

In [11]:
def train_and_log(model, run_name: str) -> None:
    """Train a model, evaluate it on the test set and log everything to MLflow."""
    with mlflow.start_run(run_name=run_name):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        rmse = root_mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        mlflow.log_params(model.get_params())
        mlflow.log_metrics({"rmse": rmse, "mae": mae})
    print(f"{run_name}: RMSE = {rmse:.3f}, MAE = {mae:.3f}")

In [12]:
train_and_log(LinearRegression(), "linear_regression")
train_and_log(RandomForestRegressor(n_estimators=100, random_state=42), "random_forest")

linear_regression: RMSE = 2.212, MAE = 1.593
random_forest: RMSE = 2.256, MAE = 1.585


## Conclusion

Runs can be compared in the MLflow UI (`uv run mlflow ui --backend-store-uri sqlite:///mlflow.db`, from the `notebooks` folder).

| Model | RMSE | MAE |
|---|---|---|
| Linear regression | 2.212 | 1.593 |
| Random forest | 2.256 | 1.585 |

The random forest does not beat the linear regression here: both models are within a few hundredths on RMSE and MAE. We keep the **linear regression** for the next steps, since it is simpler, faster to train and easier to explain.